# TFT_Implementierung
## Nicholas Katz, Simon Sarcletti
Dieses File beschreibt die Umsetzung eines Temporal Fusion Transformer für die Prognose der Gemeindebevölkerung.

### Laden notwendiger Packete

Zuerst müssen die notwendigen Packete geladen werden. Diese Umsetzung basier of PyTorch und PyTorch-Forecasting. 

Anschließend wird geprüft, ob die Grafikkarte (!) erkannt wird.

In [ ]:
import copy
from pathlib import Path
import warnings
import pickle

import numpy as np
np.Inf = np.inf # Ensure np.inf is set correctly
import pandas as pd
import torch

from pyreadr import read_r

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import EncoderNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

warnings.filterwarnings("ignore")

# --- Device and CUDA Information ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"CUDA Available: {torch.cuda.is_available()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"CUDA Version: {torch.version.cuda}")

### Laden der Daten und Vorbereitung dersselben
Das Python-Skript lädt Bevölkerungs- und statische Metadaten, führt diese zusammen und bereinigt sie. Es filtert Daten ab 2004, erstellt neue Identifikationsspalten, benennt Spalten um und löscht unnötige Informationen. Abschließend werden die Datentypen angepasst.

In [ ]:
# --- Configuration ---
# Base directory for your data
base_data_dir = Path("/data/simon/")


# --- Load Data ---
# Load population data
all_munip_pop = read_r(base_data_dir / "all_municipalities_population.RData")["all_munip_pop"]
all_munip_pop["municipality_code"] = all_munip_pop["municipality_code"].astype("int64")

# Load static metadata
static_metadata = pd.read_csv(
    base_data_dir / "static_variables.csv",
    encoding="latin-1",
    sep=";",
    decimal=",",
)

# --- Merge Data ---
merged_data = pd.merge(
    all_munip_pop,
    static_metadata,
    how="left",
    left_on="municipality_code",
    right_on="ID",
)

# --- Feature Engineering and Cleaning ---
# Create unique index
merged_data["index"] = (
    merged_data["municipality_code"].astype(str)
    + "_"
    + merged_data["sex"].round(0).astype(str)
    + "_"
    + merged_data["coarse_age_group"]
)

# Define columns to drop for better readability and maintainability
columns_to_drop = [
    "Name",
    "ID",
    "municipality_code",
    "reg_code", 
    "municipality",
    "sex",
    "population", 
]
merged_data = merged_data.drop(columns=columns_to_drop)

# Filter data by year
merged_data = merged_data[merged_data["year"] >= 2004].copy() 

# Create new 'reg_code' from 'index'
merged_data["reg_code"] = merged_data["index"].str[:3]

# Rename columns
merged_data = merged_data.rename(
    columns={"smoothed_population": "population", "coarse_age_group": "age_group"}
)

# Convert 'year' to integer type
merged_data["year"] = pd.to_numeric(merged_data["year"], downcast="integer")

# --- Define and Convert Static Feature Types ---
static_categoricals = [
    'Urban-Rural-Typologie', 'klassifikation_palme95', 'OeV-Güteklassen',
    'Bezirkshauptstadt', 'schulen_ue250', 'umkreis_schulen',
    'haltestelle_IbIII', 'haltestelle_umkreis', 'autobahnauffahrt',
    'autobahnauffahrt_umkreis', 'umkreis_einpendler', 'reg_code',
]

static_reals = [
    'Index_Pendlersaldos_2022', 'anteil_ue75_2014', 'anteil_ue75_2024',
    'durchschnittsalter', 'Jahresbruttobezug_2023',
    'anteil_frauen_1534_gesamtbevölkerung',
    'verkehrsleistung_personenkilometer_energiemosaik',
    'handelsgebaeude_1000ew_gwr', 'kulturgebaeude_1000ew_gwr',
]

# Convert static categorical columns to string type
for col in static_categoricals:
    merged_data[col] = merged_data[col].astype(str)


print("Data preprocessing complete.")
print(f"Final data shape: {merged_data.shape}")
print(merged_data.head())

### Dataset für Modell-Training
Dieses Skript definiert und erstellt die notwendigen Zeitreihen-Datensätze (TimeSeriesDataSet) für das Training und die Validierung eines Prognosemodells. Es legt Parameter wie die maximale Vorhersage- und Encoder-Länge sowie den Trainings-Trennungszeitpunkt fest. Anschließend werden DataLoader-Objekte generiert, um die Daten in Batches für das Modelltraining bereitzustellen.

* `max_encoder_length`: Maximale Länge der **historischen Eingabedaten** (Vergangenheit).
* `max_prediction_length`: Maximale Länge der **prognostizierten Ausgabedaten** (Zukunft).

Batch: Im Kontext von maschinellem Lernen und Deep Learning ist ein Batch eine kleine Untermenge des gesamten Datensatzes, die gleichzeitig verarbeitet wird.
Statt alle Trainingsdaten auf einmal in den Speicher zu laden und durch das Modell zu schicken (was bei großen Datensätzen unmöglich wäre), wird der Datensatz in diese kleineren (in diesen Fall 64) Batches aufgeteilt.

In [ ]:
# Define key parameters
max_prediction_length = 10
max_encoder_length = 10
training_cutoff = 2014

# Define the TimeSeriesDataSet for training
training = TimeSeriesDataSet(
    merged_data,
    time_idx="year",
    target="population",
    group_ids=["index"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=static_categoricals,
    static_reals=static_reals,
    time_varying_unknown_reals=["population"],
    target_normalizer=EncoderNormalizer(),
)

# Create validation dataset from training, specifying only differences
validation = TimeSeriesDataSet.from_dataset(
    training,
    merged_data,
    predict=True,
    stop_randomization=True,
    min_prediction_idx=training_cutoff + 1,
    max_prediction_length=max_prediction_length,
)

# Define batch size and create DataLoaders
batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=8)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=8)

### Initialisierung des Neuronalen Netzes
Dieses Skript initialisiert und trainiert ein neuronales Netz des Typs "Temporal Fusion Transformer" (TFT) mittels PyTorch Lightning. Zuerst werden Einstellungen für die Reproduzierbarkeit, Rückruffunktionen (z.B. Early Stopping) und Logging (TensorBoard) definiert. Anschließend wird der Trainer mit maximaler Epochenzahl, GPU-Nutzung und Gradient Clipping konfiguriert. Das TFT-Modell selbst wird mit spezifischen Hyperparametern (wie hidden_size, learning_rate und loss) initialisiert und schließlich mit den vorbereiteten Daten (Trainings- und Validierungs-Dataloader) trainiert.

In [ ]:
pl.seed_everything(42)

# --- Configure Callbacks and Logger ---
early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min"
)
lr_logger = LearningRateMonitor()
logger = TensorBoardLogger("lightning_logs")

# --- Configure Trainer ---
trainer = pl.Trainer(
    max_epochs=50,
    accelerator="gpu",
    enable_model_summary=True,
    gradient_clip_val=0.1,  # Prevent gradient divergence for RNNs
    callbacks=[lr_logger, early_stop_callback],
    logger=logger,
)

# --- Configure TemporalFusionTransformer (using the final desired parameters) ---
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,          
    attention_head_size=2,   
    dropout=0.1,            
    hidden_continuous_size=8, 
    loss=QuantileLoss(),
    optimizer="adam",
    log_interval=10,        
    reduce_on_plateau_patience=4, 
)
print(f"Number of parameters in network: {tft.size() / 1e3:.1f}k")

# --- Fit Network ---
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

### Wichtigste Hyperparameter im Kontext des TFT-Modells und PyTorch Lightning

Hyperparameter sind Einstellungen, die vor dem Training eines Modells festgelegt werden und den Lernprozess sowie die Struktur des Modells beeinflussen. Sie werden nicht vom Modell selbst gelernt.

#### Für den `pl.Trainer` (Trainingskontrolle):

* `max_epochs`:
    * **Bedeutung:** Die maximale Anzahl von Trainingsdurchläufen (Epochen), die das Modell über den gesamten Trainingsdatensatz macht.
    * **Auswirkung:** Bestimmt, wie lange das Training maximal läuft. In Kombination mit `EarlyStopping` wird das Training oft vorher beendet.
* `gradient_clip_val`:
    * **Bedeutung:** Ein Wert, der das Abschneiden (Clipping) von Gradienten steuert. Wenn der Gradient eines Parameters diesen Wert überschreitet, wird er auf diesen Wert begrenzt.
    * **Auswirkung:** Wichtig, um "exploding gradients" zu verhindern, ein Problem, bei dem Gradienten während des Trainings so groß werden, dass sie die Modellgewichte instabil machen und das Training zum Absturz bringen können. Besonders relevant bei rekurrenten neuronalen Netzen.
* `callbacks`:
    * **Bedeutung:** Eine Liste von Funktionen oder Objekten, die während des Trainings zu bestimmten Zeitpunkten ausgeführt werden.
    * **Auswirkung:** Steuern Verhaltensweisen wie:
        * `EarlyStopping`: Beendet das Training frühzeitig, wenn die Validierungsleistung über eine bestimmte Anzahl von Epochen hinweg nicht mehr besser wird, um Overfitting zu vermeiden.

#### Für das `TemporalFusionTransformer` Modell (Modellarchitektur und Lernverhalten):
* `learning_rate`:
    * **Bedeutung:** Bestimmt die Schrittgröße, mit der die Modellgewichte während der Optimierung angepasst werden.
    * **Auswirkung:** Einer der kritischsten Hyperparameter. Eine zu hohe Lernrate kann dazu führen, dass das Modell über das Optimum "hinausschießt" und nicht konvergiert. Eine zu niedrige Lernrate kann das Training sehr langsam machen oder in einem sub-optimalen lokalen Minimum stecken bleiben lassen.
* `hidden_size`:
    * **Bedeutung:** Die Größe der versteckten Schichten in den neuronalen Netzen innerhalb des TFT (z.B. für die Gating-Einheiten, Multi-Head Attention, Feed-Forward-Netzwerke).
    * **Auswirkung:** Eine größere `hidden_size` ermöglicht dem Modell, komplexere Muster zu lernen, erfordert aber mehr Rechenleistung und Speicher und erhöht das Risiko von Overfitting, wenn die Datenmenge nicht ausreicht.
* `attention_head_size`:
    * **Bedeutung:** Die Anzahl der "Aufmerksamkeitsköpfe" im Multi-Head Attention Mechanismus des TFT.
    * **Auswirkung:** Mehr Köpfe ermöglichen es dem Modell, verschiedene Aspekte der Eingabesequenz parallel zu betrachten und unterschiedliche "Beziehungen" zwischen den Zeitpunkten zu lernen.
* `dropout`:
    * **Bedeutung:** Der Anteil der Neuronen, die während des Trainings zufällig "abgeschaltet" (auf Null gesetzt) werden.
    * **Auswirkung:** Eine Regularisierungstechnik, die Overfitting reduziert, indem sie das Modell zwingt, robustere und weniger spezifische Merkmale zu lernen. Werte zwischen 0.1 und 0.3 sind üblich.
* `reduce_on_plateau_patience`:
    * **Bedeutung:** Die Anzahl der Epochen, die das Modell warten soll, ohne dass sich die Validierungsverlust verbessert, bevor die Lernrate automatisch reduziert wird.
    * **Auswirkung:** Eine Lernraten-Scheduling-Strategie, die hilft, das Training fortzusetzen, wenn das Modell auf einem Plateau stagniert, indem es die Lernrate verkleinert und feinere Anpassungen ermöglicht.

### Hyperparameter-Tuning
Dieses Skript führt eine Hyperparameter-Optimierung für das Temporal Fusion Transformer (TFT)-Modell durch, um die besten Einstellungen (wie Lernrate, versteckte Größe) zu finden. Die Ergebnisse dieser Optimierungsstudie werden gespeichert. Anschließend wird das beste Modell aus einem zuvor gespeicherten Checkpoint geladen, um es für weitere Schritte wie Vorhersagen zu nutzen.

In [ ]:
# Define base path for consistency
BASE_SAVE_PATH = Path("/home/v18y97/mt_pop_forecast/")
MODEL_TUNING_PATH = BASE_SAVE_PATH / "tft_for_prediction_tuning"
STUDY_SAVE_PATH = BASE_SAVE_PATH / "prediction_study.pkl"

study = optimize_hyperparameters(
    train_dataloader,
    val_dataloader,
    model_path=str(MODEL_TUNING_PATH), # Convert Path to string for model_path
    n_trials=200,
    max_epochs=50,
    gradient_clip_val_range=(0.01, 1.0),
    hidden_size_range=(8, 128),
    hidden_continuous_size_range=(8, 128),
    attention_head_size_range=(1, 4),
    learning_rate_range=(0.001, 0.1),
    dropout_range=(0.1, 0.3),
    trainer_kwargs=dict(limit_train_batches=30),
    reduce_on_plateau_patience=4,
    use_learning_rate_finder=False,
)

# Save study results
with open(STUDY_SAVE_PATH, "wb") as fout:
    pickle.dump(study, fout)
print(f"Hyperparameter optimization study saved to: {STUDY_SAVE_PATH}")

## Vorhersage

### Vorbereitung der Daten
Dieses Skript bereitet Daten für Zeitreihen-Vorhersagen vor: Es entnimmt die letzten Jahre an Vergangenheitsdaten (Encoder-Daten) und generiert zukünftige Zeitpunkte basierend auf den aktuellsten Daten (Decoder-Daten). Beide Teile werden dann zu einem neuen Datensatz zusammengeführt, der für Prognosen bereit ist. Da nur statische Variablen benutzt werden, können sie einfach fortgeschrieben werden.

In [ ]:
# Parameters for look-back and look-ahead
max_encoder_length = 25
max_prediction_length = 11

# 1. Extract historical data (encoder part)
# Filters to include data from the last `max_encoder_length` years relative to the latest year.
encoder_data = merged_data[merged_data.year > merged_data.year.max() - max_encoder_length].copy()

# 2. Generate future data points (decoder part)
# Get the most recent year's data for each group
last_data_point = merged_data[merged_data.year == merged_data.year.max()].copy()

# Create future time steps by incrementing the year for each prediction length
decoder_data = pd.concat(
    [
        last_data_point.assign(year=lambda x, i=i: x['year'] + i)
        for i in range(1, max_prediction_length + 1)
    ],
    ignore_index=True,
)

# 3. Combine historical and future data
new_prediction_data = pd.concat([encoder_data, decoder_data], ignore_index=True)

# 4. Ensure static categorical columns remain string type
for col in static_categoricals:
    new_prediction_data[col] = new_prediction_data[col].astype(str)

print("Data preprocessing for prediction complete.")
print(f"Final data shape: {new_prediction_data.shape}")
print(new_prediction_data.head())

### Vorhersage auf Basis der Vorhersagedaten
Dieses Skript führt die tatsächliche Vorhersage mit dem trainierten Modell durch. Die generierten, mehrdimensionalen Rohprognosen werden anschließend in ein flaches (long-format) DataFrame umgewandelt, das die ursprünglichen Indizes, die prognostizierten Jahre, die Quantile und die Vorhersagewerte enthält. Das fertige DataFrame wird dann als CSV-Datei gespeichert.

In [ ]:
# --- Load Best Model ---
best_model_path = trainer.checkpoint_callback.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# --- Actual Prediction ---
new_raw_predictions = best_tft.predict(
    new_prediction_data,
    mode="raw",
    return_index=True, 
    trainer_kwargs=dict(accelerator="gpu"),
)

# Extract predictions array and its shape
predictions_array = new_raw_predictions.output.prediction.detach().cpu().numpy()
n_samples, n_steps, n_quantiles = predictions_array.shape
print(f"Shape of predictions: {predictions_array.shape}")

# Define quantiles (consistent with your previous code)
quantiles = ["0.01", "0.1", "0.25", "0.5", "0.75", "0.9", "0.99"]

# --- Reshape Predictions into Long Format DataFrame ---
# Create a MultiIndex for the resulting DataFrame columns: group_id, prediction_year, quantile
# This directly generates the correct combinations for all rows.
full_prediction_index = pd.MultiIndex.from_product(
    [
        new_raw_predictions.index["index"].unique(),  # All unique group IDs
        range(2025, 2025 + n_steps),                 # Fixed range of prediction years (e.g., 2025-2035)
        quantiles                                    # All quantiles
    ],
    names=["original_index", "year", "quantile"]
)

# Create the DataFrame directly from the flattened predictions and the generated MultiIndex
df_long = pd.DataFrame(
    predictions_array.flatten(),
    index=full_prediction_index,
    columns=["prediction"]
).reset_index() # Convert the MultiIndex levels into regular DataFrame columns

# --- Save Results ---
df_long.to_csv("/home/v18y97/mt_pop_forecast/tft_prediction_2025-2035.csv", index=False)

print(f"Predictions saved to: /home/v18y97/mt_pop_forecast/tft_prediction_2025-2035.csv")